## NN 다중분류
- 아이리스 데이터를 통해 이진분로

## 1. 데이터 준비

In [ ]:
!wget https://raw.githubusercontent.com/devdio/flyai_datasets/main/iris.csv

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42

In [ ]:
file_path = 'iris.csv'
iris = pd.read_csv(file_path)
iris.shape

In [ ]:
col_dict = {col: col.lower().replace(' ', '_' ) for col in iris.columns}
col_dict

In [ ]:
iris.rename(columns=col_dict, inplace=True)

In [ ]:
# 결측치
iris.isna().sum(axis=0)

# 중복치
iris.duplicated().sum()

In [ ]:
df = iris.copy()

## 2. 트레인 테스트 분리

In [ ]:
from sklearn.model_selection import train_test_split
SEED = 42
train, test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['species'])

train.shape, test.shape

In [ ]:
train.isna().sum()
train

In [ ]:
train = train.dropna(axis=0)
train

In [ ]:
X_train = train.drop('species', axis=1)
y_train = train['species']

X_train.shape, y_train.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_e = le.fit_transform(y_train)
y_train_e

In [ ]:
from keras.utils import to_categorical

y_train_o = to_categorical(y_train_e)
y_train_o

In [ ]:
from sklearn.preprocessing import RobustScaler

rs = RobustScaler()
X_train_s = rs.fit_transform(X_train)
X_train_s

=====================================================

## 3. 모델 학습

In [ ]:
print(X_train_s.shape, y_train.shape)
print(type(X_train_s), type(y_train_o))


In [ ]:
import keras
from keras import layers
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(4,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(3, activation='softmax') # 다중 분류는 시그모이드로 가라 -> 확률로 나옴
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
X_train_s.shape, y_train_o.shape

In [ ]:
EPOCH = 100
BATCH_SIZE = 16
history = model.fit(
    X_train_s, y_train_o,
    epochs=EPOCH,
    batch_size=BATCH_SIZE,
    validation_split=0.2
)

In [ ]:
def plot_history(history):
    hist = pd.DataFrame(history.history)
    hist['epoch'] = history.epoch

    plt.figure(figsize=(16, 8))
    plt.subplot(1, 2, 1)
    plt.xlabel('epochs')
    plt.ylabel('loss')
    plt.plot(hist['epoch'], hist['loss'], label='train loss')
    plt.plot(hist['epoch'], hist['val_loss'], label='val loss')
    plt.title('Loss Curve')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.xlabel('epochs')
    plt.ylabel('accuracy')
    plt.plot(hist['epoch'], hist['accuracy'], label='train accuracy')
    plt.plot(hist['epoch'], hist['val_accuracy'], label='val accuracy')
    plt.title('Accuracy Curve')
    plt.legend()
    plt.show()

In [ ]:
plot_history(history)

## 4. 검증

In [ ]:
test = test.dropna(axis=0)
test

X_test = test.drop('species', axis=1)
y_test = test['species']

X_test.shape, y_test.shape

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_test_e = le.fit_transform(y_test)
y_test_e

from sklearn.preprocessing import RobustScaler

rs = RobustScaler()
X_test_s = rs.fit_transform(X_test)
X_test_s


In [ ]:
y_pred = model.predict(X_test_s)
y_pred

In [ ]:
import numpy as np

y_pred = np.argmax(y_pred, axis=1)
y_pred

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.metrics import confusion_matrix

def print_metrics(y_true, y_pred, ave='binary'):
  print('accuracy:', accuracy_score(y_test_e, y_pred))
  print('recall:', recall_score(y_test_e, y_pred, average=ave))
  print('precision:', precision_score(y_test_e, y_pred, average=ave))
  print('f1 :', f1_score(y_test_e, y_pred, average=ave))

  clm = confusion_matrix(y_test_e, y_pred)
  s = sns.heatmap(clm, annot=True, cmap='Blues', fmt='d', cbar=False)
  s.set(xlabel='Predicted', ylabel='Actual')

In [ ]:
print_metrics(y_test_e, y_pred, ave='macro')